# Week 4 — Customer Order Insights



In [2]:
!pip -q install pyspark==3.5.1 delta-spark==3.1.0

In [3]:
from google.colab import files
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, upper, lower, to_date, when, count, sum as spark_sum, avg, datediff, current_date
from delta import configure_spark_with_delta_pip
import os
import shutil

In [4]:
uploaded = files.upload()
required_files = ('customers.csv', 'orders.csv')
missing_files = set(required_files) - set(uploaded.keys())
if missing_files:
    raise ValueError(f"Upload these files: {sorted(missing_files)}")

Saving customers.csv to customers.csv
Saving orders.csv to orders.csv


In [15]:
builder = SparkSession.builder \
    .appName("Week4CustomerOrderETL") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

## 1. Load source data

In [9]:
reference_raw = spark.read.option("header", True).csv("/content/customers.csv")
raw = spark.read.option("header", True).csv("/content/orders.csv")
reference = reference_raw
customers_raw = reference_raw
raw.show(5, truncate=False)
reference_raw.show(5, truncate=False)

+--------+-----------+----------+----------------------+--------------------+------------+----------+
|order_id|customer_id|order_date|expected_delivery_date|actual_delivery_date|order_amount|status    |
+--------+-----------+----------+----------------------+--------------------+------------+----------+
|ORD001  |C001       |2026-01-01|2026-01-06            |2026-01-06          |587         |SHIPPED   |
|ORD002  |C002       |2026-01-02|2026-01-07            |2026-01-07          |724         |PROCESSING|
|ORD003  |C003       |2026-01-03|2026-01-08            |2026-01-08          |861         |CANCELLED |
|ORD004  |C004       |2026-01-04|2026-01-09            |2026-01-11          |998         |DELIVERED |
|ORD005  |C005       |2026-01-05|2026-01-10            |NULL                |1135        |SHIPPED   |
+--------+-----------+----------+----------------------+--------------------+------------+----------+
only showing top 5 rows

+-----------+-------------+----------+-------------------

## 2. Clean the reference data

In [10]:
reference = reference.select(trim("customer_id").alias("customer_id"), trim("customer_name").alias("customer_name"), trim("city").alias("city"), lower(trim("email")).alias("email")).filter((col("customer_id") != "") & col("customer_id").isNotNull() & (col("customer_name") != "")).dropDuplicates(["customer_id"])
reference.show(truncate=False)
print('Reference rows after cleaning:', reference.count())

+-----------+---------------+----------+--------------------------+
|customer_id|customer_name  |city      |email                     |
+-----------+---------------+----------+--------------------------+
|C001       |Aarav Stores   |Chennai   |aaravstores@example.com   |
|C002       |Metro Mart     |Mumbai    |metromart@example.com     |
|C003       |Fresh Foods    |Hyderabad |freshfoods@example.com    |
|C004       |Quick Shop     |Bengaluru |quickshop@example.com     |
|C005       |Green Basket   |Coimbatore|greenbasket@example.com   |
|C006       |City Bazaar    |Madurai   |citybazaar@example.com    |
|C007       |Daily Needs    |Pune      |dailyneeds@example.com    |
|C008       |Prime Retail   |Kochi     |primeretail@example.com   |
|C009       |Sunrise Traders|Salem     |sunrisetraders@example.com|
|C010       |Value Point    |Trichy    |valuepoint@example.com    |
|C011       |North Star     |Delhi     |northstar@example.com     |
|C012       |Ocean Mart     |Vizag     |oceanmar

## 3. Clean and transform transactional data

In [11]:
data = raw.select(trim("order_id").alias("order_id"), trim("customer_id").alias("customer_id"), to_date(trim("order_date")).alias("order_date"), to_date(trim("expected_delivery_date")).alias("expected_delivery_date"), to_date(trim("actual_delivery_date")).alias("actual_delivery_date"), col("order_amount").cast("double").alias("order_amount"), upper(trim("status")).alias("status"))
data = data.filter((col("order_id") != "") & col("order_id").isNotNull() & (col("customer_id") != "") & col("order_date").isNotNull() & col("expected_delivery_date").isNotNull() & (col("order_amount") > 0)).dropDuplicates(["order_id"])
data = data.withColumn("status", when(col("status").isNull() | (col("status") == ""), "PENDING").otherwise(col("status")))
data.show(truncate=False)
print('Transaction rows after cleaning:', data.count())

+--------+-----------+----------+----------------------+--------------------+------------+----------+
|order_id|customer_id|order_date|expected_delivery_date|actual_delivery_date|order_amount|status    |
+--------+-----------+----------+----------------------+--------------------+------------+----------+
|ORD001  |C001       |2026-01-01|2026-01-06            |2026-01-06          |587.0       |SHIPPED   |
|ORD002  |C002       |2026-01-02|2026-01-07            |2026-01-07          |724.0       |PROCESSING|
|ORD003  |C003       |2026-01-03|2026-01-08            |2026-01-08          |861.0       |CANCELLED |
|ORD004  |C004       |2026-01-04|2026-01-09            |2026-01-11          |998.0       |DELIVERED |
|ORD005  |C005       |2026-01-05|2026-01-10            |NULL                |1135.0      |SHIPPED   |
|ORD006  |C006       |2026-01-06|2026-01-11            |2026-01-11          |1272.0      |PROCESSING|
|ORD007  |C007       |2026-01-07|2026-01-12            |2026-01-12          |1409.

## 4. Join and derive business metrics

In [12]:
details = data.join(reference, "customer_id", "inner").withColumn("delay_days", when(col("actual_delivery_date").isNotNull(), datediff(col("actual_delivery_date"), col("expected_delivery_date"))).otherwise(datediff(current_date(), col("expected_delivery_date")))).withColumn("is_delayed", col("delay_days") > 0)
summary = details.groupBy("city").agg(count("order_id").alias("total_orders"), spark_sum(when(col("is_delayed"),1).otherwise(0)).alias("delayed_orders"), spark_sum("order_amount").alias("total_order_value"), avg("delay_days").alias("average_delay_days"))
details.show(10, truncate=False)
summary.show(truncate=False)

+-----------+--------+----------+----------------------+--------------------+------------+----------+---------------+----------+--------------------------+----------+----------+
|customer_id|order_id|order_date|expected_delivery_date|actual_delivery_date|order_amount|status    |customer_name  |city      |email                     |delay_days|is_delayed|
+-----------+--------+----------+----------------------+--------------------+------------+----------+---------------+----------+--------------------------+----------+----------+
|C001       |ORD001  |2026-01-01|2026-01-06            |2026-01-06          |587.0       |SHIPPED   |Aarav Stores   |Chennai   |aaravstores@example.com   |0         |false     |
|C002       |ORD002  |2026-01-02|2026-01-07            |2026-01-07          |724.0       |PROCESSING|Metro Mart     |Mumbai    |metromart@example.com     |0         |false     |
|C003       |ORD003  |2026-01-03|2026-01-08            |2026-01-08          |861.0       |CANCELLED |Fresh Foo

## 5. Write cleaned data to CSV and Delta Lake

In [22]:
output_root = "/content/week4_customer_order_insights_outputs"
os.makedirs(output_root, exist_ok=True)
data.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{output_root}/cleaned_data_csv")
details.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{output_root}/joined_details_csv")
summary.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{output_root}/business_summary_csv")
data.write.format("delta").mode("overwrite").save(f"{output_root}/cleaned_data_delta")
details.write.format("delta").mode("overwrite").save(f"{output_root}/joined_details_delta")
summary.write.format("delta").mode("overwrite").save(f"{output_root}/business_summary_delta")

## 6. Read back and validate Delta data

In [22]:
clean_delta = spark.read.format("delta").load(f"{output_root}/cleaned_data_delta")
detail_delta = spark.read.format("delta").load(f"{output_root}/joined_details_delta")
summary_delta = spark.read.format("delta").load(f"{output_root}/business_summary_delta")
print("Clean rows:", clean_delta.count())
print("Joined rows:", detail_delta.count())
print("Summary rows:", summary_delta.count())
clean_delta.show(5, truncate=False)

## 7. Download generated deliverables

In [23]:
shutil.make_archive("/content/week4_customer_order_insights_outputs", "zip", output_root)
files.download("/content/week4_customer_order_insights_outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
spark.stop()